# Simulating FACS

In this notebook, I'll attempt to model the statistical processes of a FACS experiment. This model should connect biophysical phenotypes to the probability of being sequenced. This model is the product of hours of my interrogating Gemini. While I lack the statistical background to have come up with this on my own, I intend to use this as a baseline for simulation and identify problems as needed.

## Model Derivation

### Biophysics

For a protein to be detected in a sorted population, it must first fold, then be secreted, then bind the ligand at a given concentration. 

The probability of a protein folding and binding have theoretical equations, but the probability of being secreted given that it folds is more complex and would need empirical modeling for a given protein. 

For now, we will assume that the probability of being secreted is constant for all variants, so the protein stability is the main factor driving expression.

$P_{fold}$ is governed by the Boltzmann distribution:

$$
P_{fold,v} = \frac{1}{1 + e^{\Delta G_{fold,v} / RT}}
$$

$P_{bind}$ is governed by the Hill-Langmuir equation:

$$
P_{bind,v} = \frac{[L]^{n_{Hill}}}{[L]^{n_{Hill}} + (K_{d,v})^{n_{Hill}}}
$$

### Sorting

But FACS doesn't look at expression or binding directly, it looks at fluorescence intensity. Let's convert these physical expectations to fluorescence.

We can expect the mean receptor fluorescence intensity of a cell carrying variant $v$, $\mu_{E,v}$ to be proportional to the probability of folding scaled by its surface expression capacity (which we'll call $S_{exp}$), and mean ligand fluorescence intensity, $\mu_{B,v}$, to be proportional to the probability of folding and binding

$$
\mu_{E,v} \propto S_{exp} \times P_{fold,v}
$$
$$
\mu_{B,v} \propto \mu_{E,v} \times P_{bind,v}
$$

Note, however, that these are means. There's biological stochasticity here that we need to deal with.

How can we represent that biological stochasticity? Looking at the histograms in Rim et al., 2024 for single variants suggests that, excepting the cluster of cells not expressing anything, those that are expressing do so with a log-normal distribution.

What is a log-normal distribution? It's when the natural log of the values follows a normal distribution. It's often the assumption for cellular expression data [Furusawa et al., 2005](https://pmc.ncbi.nlm.nih.gov/articles/PMC5036630/).

We can then represent the receptor expression signal, $E_{cell}$, and the ligand binding signal, $B_{cell}$ for each cell as follows.
$$
E_{cell} \sim \text{LogNormal}(\ln{(\mu_{E,v})}, \sigma_{bio}^2)
$$

$$
B_{cell} \approx E_{cell} \times P_{bind,v}
$$

As a first approximation, we are going to assume that binding noise is negligible compared to expression noise since each cell typically displays 1e4 - 1e5 copies of an Aga2p-fused protein. However, it will be worth it eventually to model ligand depletion, which becomes important as the ligand concentration reduces.

To suvive the sort, $E_{cell} > Gate_{receptor}$ AND $B_{cell} > Gate_{ligand}$. At first, this looks like a 2-D problem, but as you can see from above, $B_{cell}$ is dependent on $E_{cell}$. 

$$
E_{cell} \approx  \frac{B_{cell}}{P_{bind,v}}
$$

If you know $E_{cell}$ and $P_{bind,v}$, as we will in our simulation, you can determine $B_{cell}$, so you don't need to consider it separately, you just need to cross the harder threshold.

For example, say your gate says that receptor expression signal must be at least 100 units. Then, it says your binding signal must be at least 500 units. If that variant is a weak binder ($P_{bind} = 0.1$), it effectively says you need 5,000 units of expression to generate that signal. In this case, the effective threshold for sorting is the 5,000 units for expression.

The opposite scenario would be possible if your receptor expression signal must be at least 500 units and your binding signal 100. If you have a super binder with 100% occupancy, 100 units of receptor is enough to cross the binding threshold, but it would fail to cross the receptor expression signal.

So, the sorting threshold is:
$$
T_{effective} = \max(Gate_{exp}, \frac{Gate_{bind}}{P_{bind,v}})
$$

Now, to relate our predicted mean from physics to the probability of crossing the sorting threshold, we look at the area under the probability density function for a Log-Normal distribution that crosses the threshold (limits T to inf). This would be the integral of the probability density function, but there's a precalculated function we can use called the "Cumulative Distribution Function" ($\Phi$), given the z-score. It's defined as follows:

$$
\Phi(z) = \int\limits_{-\infty}^z \text{PDF}(t)dt
$$

So it's the area from the left to your z-score. But, since we are working with a normal distribution (when in log space), which is symmetric, the area to the right of the z-score (above our threshold) is equal to the area to the left of the negative z-score.

To get a z-score, we also need to know how the standard deviation of FACS measurements for a clonal population. This can be obtained directly from the control data.

Therefore, we can get the probability that a cell expressing v is sorted as:

$$
P_{sort} = \Phi(\frac{-(\ln{(T_{effective})} - \ln{(\mu_{E,v})})}{\sigma_{FACS}})
$$

Finally, to turn this continuous probability into a discrete number of cells, $C_v$, we draw a random sample from a binomial distribution (because a cell can be either sorted or not) from the input number of cells, $N_{v,0}$.

$$
C_{v} \sim \text{Binomial}(N_{v,0}, P_{sort}(v))
$$

### Sequencing

Okay, so we have some cells in our tube now. What's the probability the variant carried by these yeasty bois get sequenced?

First, we need to extract the plasmids and transfer them to the PCR reaction tube. Let $\rho$ be the plasmid copy number, and $\epsilon_{rec}$ be the efficiency of extraction and transfer to the PCR reaction tube (note: I'm clumping these together because I wasn't able to determine a plasmid extraction efficiency with precision using NanoDrop concentrations; the best I could do is just assume 100% extraction then use the fraction of purified plasmid volume used as the template, 1 uL used as template / 10 uL total plasmid volume).

Again, plasmids will either make it into the tube or not, so the number of template molecules entering PCR, $T_v$, is:

$$
T_v \sim \text{Binomial}(C_v \cdot \rho, \epsilon_{rec})
$$

A second note: I just checked the Yeast Miniprep kit website, and yield is very, very low. They say yield is "typically between 0.01-0.3 ng for most 2 µ based plasmids from 1.5 ml overnight cultures". I revised my library_analysis.ipynb to check whether this could be a worse bottleneck than sequencing and confirmed that sequencing was still the bottleneck, but plasmid prep and transfer reduced library diversity to within an order of magnitude of the read depth. So, $\epsilon_{rec}$ is a key parameter to get right.

The next step is to model the PCR amplification process. This is important because PCR jackpotting is a likely major source of noise. This is when there are few template molecules for a variant, so whether or not a molecule gets amplified in an early cycle of PCR has a large impact on the final number of copies.

The ideal equation relating the number of input templates to output amplicons with amplification efficiency $\epsilon_{amp}$ and $C$ cycles would be:

$$
A_v = T_v \cdot (1 + \epsilon_{amp})^C
$$

However, there is noise in this process, of course.

$$
A_v = T_v \cdot (1 + \epsilon_{amp})^C \cdot Noise
$$

where Noise is dependent on $T_v$ and relates to jackpotting. When $T_v$ is high, the noise decreases as whether an individual template was amplified or not doesn't make much of a difference. When $T_v$ is low, it makes a big difference, so noise is high.

The distribution to model this sort of process is a Gamma distribution, which is the sum of independent exponential growth processes. Gamma distributions are defined by two parameters: shape, k, and scale $\theta$. 

k controls the skew. As k decreases, the Gamma distribution becomes more exponential-like, as we would expect the distribution of $A_v$ when $T_v$ is low. As k increases, it becomes more Gaussian.

We can therefore define k as proportional to $T_v$, with some noise factor that would ideally be fit empirically.

$$
k = T_v \times \beta_{PCR}
$$

To get the scale, we use the fact that the mean of a Gamma distribution is $k \times \theta$. We defined what the mean should be biologically above (assuming we don't run out of dNTPs in the tube). To solve for $\theta$, we can set them equal to each other and solve.

$$
T_v \times (1 + \epsilon_{amp})^C = T_v \times \beta_{PCR} \times \theta
$$

$$
\theta = \frac{(1 + \epsilon_{amp})^C}{\beta_{PCR}}
$$

Giving us that the number of amplicons for variant v is:

$$
A_v \sim \text{Gamma}(T_v \times \beta_{PCR}, \frac{(1 + \epsilon_{amp})^C}{\beta_{PCR}})
$$

These amplicons now get sent to the sequencer, which is where the biggest bottleneck typically is. There are two sources of error we'll consider here. First is the identity error, representing miscalled bases. This is important where a rare variant count is overinflated by being close in identity to a common variant.

To model this, we can define an error matrix, $E$ where $E_{ij} = P(\text{Read as i |Actually j})$. The effective abundances of $A'_v$ including false positives from other variants is therefore:

$$
A'_v = \sum_j E_{vj}A_j
$$

This one we will treat as theoretical for now because it would require modeling the full N x N error matrix for all variants and would be computationally expensive to simulate.

The second, biggest source of error is read sampling. The final observed count, $n_v$ follows a multinomial distribution, where each variant is a choice that can be drawn, with the read depth, $N_{reads}$ being the number of drawings and the frequency of the variant (number of amplicons for variant divided by total amplicons) in the pool being the probability.

$$
n_v \sim \text{Multinomial}(N_{reads}, \frac{A'_v}{\sum_k A'_k})
$$

And that's it! That's the model translating the biophysics and experimental parameters into read counts from NGS.

Now, let's see if we can translate this to code...

## Simulation

### Building the simulator

The first thing built in the model above was the biophysics, but even before that, there's an outgrowth step preceding induction. We assume that this amplifies the provided counts proportionally to a total number of 10^8 cells, but the specific number will be a user parameter.

In [1]:
def scale_library_to_physical_count(df, num_cells_sorted):
        current_total = df['count'].sum()
        scaling_factor = num_cells_sorted / current_total
        
        # Scale and convert to integer
        df['count'] = (df['count'] * scaling_factor).astype(int)
        
        return df

First, we simulate the biophysics to calculate the probabilities of folding and being occupied by a ligand. The function will expect ddG values (difference in dG of variant compared to WT) for folding and binding the target ligand.

In [2]:
import numpy as np

def simulate_biophysics(
        df,
        RT,
        wt_dG_fold,
        wt_dG_bind,
        ligand_conc,
        hill_coeff
):
    # Calculate aboslute energies from wild-type and ddG values
    df['dG_fold'] = wt_dG_fold + df['ddG_fold']
    df['p_fold'] = 1.0 / (1.0 + np.exp(df['dG_fold'] / RT))

    df['dG_bind'] = wt_dG_bind + df['ddG_bind']
    df['Kd_variant'] = np.exp(df['dG_bind'] / RT)
    df['occupancy'] = (ligand_conc)**hill_coeff / ((ligand_conc)**hill_coeff + (df['Kd_variant'])**hill_coeff)
    return df

Next we simulate the FACS sorting. We represent the threshold as a fraction of the theoretical maximum signal, where $P_{fold}$ = 1 and occupancy = 1. A threshold of 0.1 is 10% of the maximum.

In [3]:
from scipy.stats import norm

def simulate_facs(
        df,
        t_exp,
        t_bind,
        facs_noise,
):
    # Determine effective threshold
    with np.errstate(divide='ignore'): # handle division by 0
        req_expression_for_binding = t_bind / df['occupancy']
    
    t_effective = np.maximum(t_exp, req_expression_for_binding)

    # Log transform to normal distribution
    mu_log = np.log(df['p_fold'] + 1e-12)
    thresh_log = np.log(t_effective + 1e-12)

    # Calculate negative z-scores
    z_scores = -(thresh_log - mu_log) / facs_noise

    # Use CDF to calculate probability of sorting
    df['p_sort'] = norm.cdf(z_scores)

    # Handle when occupancy is 0, which produces infinite t_effective
    df.loc[np.isinf(t_effective), 'p_sort'] = 0.0

    # Execute the sort
    df['n_cells_sorted'] = np.random.binomial(
        n=df['count'].values.astype(int),
        p=df['p_sort'].values
    )

    return df


Our virtual cells are now in virtual tubes, so let's extract their virtual plasmids and put them into a virtual PCR tube. There's a few things that I'm adding to the model here because I'm realizing the bundled recovery efficiency parameter isn't practical. Sorted cells are grown to saturation then sampled for miniprep. I'll assume the Zymo kit efficiencies for miniprep. There's some elution volume, then some fraction of that elutant is used as PCR template. So, this simulation will use that info to calculate the input template molecules. 

In [4]:
def simulate_template_extraction(
        df,
        miniprep_input_cells,
        plasmid_copy_number,
        miniprep_efficiency,
        elution_vol,
        template_vol,
        source_col
):
    # Calculate variant frequencies in outgrowth culture
    total_sorted = df[source_col].sum()
    sorted_freqs = df[source_col].values / total_sorted

    # Sample from outgrowth culture
    df['n_cells_pelleted'] = np.random.multinomial(n=int(miniprep_input_cells), pvals=sorted_freqs)

    # Determine plasmids availble for isolation
    potential_plasmids = df['n_cells_pelleted'].values * plasmid_copy_number

    # Sample from those plasmids according to kit efficiency
    df['n_plasmids_eluted'] = np.random.binomial(
        n=potential_plasmids.astype(np.int64),
        p=miniprep_efficiency
    )

    # Aliquot to PCR tube
    vol_fraction = template_vol / elution_vol
    df['Tv'] = np.random.binomial(
        n=df['n_plasmids_eluted'].values.astype(np.int64),
        p=vol_fraction
    )

    return df

We have templates in the PCR tube, so let's do PCR!

In [5]:
def simulate_PCR(
        df,
        pcr_cycles,
        pcr_efficiency,
        pcr_noise
):
    # Determine the Gamma function parameters
    mean_gain = (1 + pcr_efficiency) ** pcr_cycles
    shape = df['Tv'].values * pcr_noise
    scale = mean_gain / pcr_noise

    # Sample from distribution for amplicon counts
    amplicons = np.zeros(len(df))
    mask = shape > 0
    if np.any(mask):
        amplicons[mask] = np.random.gamma(shape[mask], scale)
    df['n_amplicons'] = amplicons

    return df

Last and technically least, our sequencing bottleneck.

In [6]:
def simulate_sequencing(
        df,
        seq_depth,
        source
):
    # Calculate amplicon frequencies
    total_molecules = df['n_amplicons'].sum()
    amp_freqs = df['n_amplicons'].values / total_molecules

    # Sample
    df[f'{source}_read_counts'] = np.random.multinomial(n=seq_depth, pvals=amp_freqs)

    return df

And that's it! It's much easier than I thought it would be. Now, let's put everything together into a simulation function!

In [7]:
def simulate_experiment(
        df,
        num_cells_sorted,
        RT,
        wt_dG_fold,
        wt_dG_bind,
        ligand_conc,
        hill_coeff,
        t_exp,
        t_bind,
        facs_noise,
        miniprep_input_cells,
        plasmid_copy_number,
        miniprep_efficiency,
        elution_vol,
        template_vol,
        pcr_cycles,
        pcr_efficiency,
        pcr_noise,
        seq_depth
):
    # Outgrowth post-transformation
    df = scale_library_to_physical_count(
        df,
        num_cells_sorted
    )

    # Sequence library
    df = simulate_template_extraction(
        df,
        miniprep_input_cells,
        plasmid_copy_number,
        miniprep_efficiency,
        elution_vol,
        template_vol,
        source_col="count"
    )
    df = simulate_PCR(
        df,
        pcr_cycles,
        pcr_efficiency,
        pcr_noise
        )
    df = simulate_sequencing(
        df,
        seq_depth,
        source='lib'
        )

    # Expose to ligand
    df = simulate_biophysics(
        df,
        RT,
        wt_dG_fold,
        wt_dG_bind,
        ligand_conc,
        hill_coeff
        )
    
    # Sort
    df = simulate_facs(
        df,
        t_exp,
        t_bind,
        facs_noise
        )
    
    # Sequence sorted population
    df = simulate_template_extraction(
        df,
        miniprep_input_cells,
        plasmid_copy_number,
        miniprep_efficiency,
        elution_vol,
        template_vol,
        source_col="n_cells_sorted"
        )
    df = simulate_PCR(
        df,
        pcr_cycles,
        pcr_efficiency,
        pcr_noise
        )
    df = simulate_sequencing(
        df,
        seq_depth,
        source='sort'
        )
    
    return df

### Ground-truth data prep

I'll try my simulation on different datasets. One will be a fully synthetic dataset that I made with ESM2. I'd also like to try the GB1 mutant and spike protein mutant datasets as ground truth.

Ground truth dataset should have columns count, ddG_fold, ddG_bind.

WT dG values for GB1:  
- fold: -4.8 kcal/mol
- bind: -9.0 kcal/mol

In [8]:
import pandas as pd

#scored_df = pd.read_parquet('outputs/synthetic_library_scored.parquet')
#scored_df = scored_df.sample(n=35_000)

gb1_df = pd.read_csv("gb1_full_library_ddG.csv")

In [17]:
params = {
    'num_cells_sorted': 1e8,
    'RT': 0.001987 * 298.15,
    'wt_dG_fold': -4.8,
    'wt_dG_bind': -9.0,
    'ligand_conc': 100e-9,
    'hill_coeff': 1.0,
    't_exp': 0.9,
    't_bind': 0.95,
    'facs_noise': 0.5,
    'miniprep_input_cells': 4e7,
    'plasmid_copy_number': 1,
    'miniprep_efficiency': 0.0016,
    'elution_vol': 10,
    'template_vol': 1,
    'pcr_cycles': 25,
    'pcr_efficiency': 0.9,
    'pcr_noise': 1.5,
    'seq_depth': 350_000
}

exp_df = simulate_experiment(gb1_df, **params).copy()

In [18]:
exp_df.n_cells_sorted.sum() / exp_df['count'].sum()

np.float64(0.01564544)

In [19]:
exp_df

,aa_substitutions,ddG_fold,ddG_bind,count,n_cells_pelleted,n_plasmids_eluted,Tv,n_amplicons,lib_read_counts,dG_fold,p_fold,dG_bind,Kd_variant,occupancy,p_sort,n_cells_sorted,sort_read_counts
0,NaN,0.000000,0.000000,1227,194,0,0,0.0,123,-4.800000,0.999697,-9.000000,2.525085e-07,0.283681,7.806721e-03,7,0
1,Q2A,1.977038,-0.354784,47,72,0,0,0.0,0,-2.822962,0.991550,-9.354784,1.387366e-07,0.418872,4.898593e-02,3,0
2,Q2C,1.926444,0.097025,12,0,0,0,0.0,0,-2.873556,0.992236,-8.902975,2.974428e-07,0.251609,3.761467e-03,0,0
3,Q2D,2.676374,0.199019,54,0,0,0,0.0,0,-2.123626,0.973002,-8.800981,3.533238e-07,0.220593,1.464829e-03,0,0
4,Q2E,1.126964,0.185110,263,0,0,0,0.0,0,-3.673036,0.997975,-8.814890,3.451253e-07,0.224656,1.939488e-03,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537126,T55Y E56S,2.206998,1.931861,238,0,0,0,0.0,0,-2.593002,0.987592,-7.068139,6.583940e-06,0.014961,4.146893e-17,0,0
537127,T55Y E56T,2.441320,1.787153,1216,0,0,0,0.0,0,-2.358680,0.981682,-7.212847,5.157073e-06,0.019022,1.940591e-15,0,0
537128,T55Y E56V,2.289906,2.214742,63,0,0,0,0.0,0,-2.510094,0.985755,-6.785258,1.061353e-05,0.009334,8.939990e-21,0,0
537129,T55Y E56W,1.240114,2.007181,51,0,0,0,0.0,0,-3.559886,0.997550,-6.992819,7.476554e-06,0.013199,5.761690e-18,0,0


In [ ]:
wt_lib = exp_df.loc[exp_df.aa_substitutions == '', 'lib_read_counts'].sum()
wt_sort = exp_df.loc[exp_df.aa_substitutions == '', 'sort_read_counts'].sum()
exp_df['enrichment'] = np.log2(((exp_df.sort_read_counts + 1) / (wt_sort + 1))/((exp_df.lib_read_counts + 1) / (wt_lib + 1)))
#exp_df['enrichment'] = np.log2(((exp_df.sort_read_counts + 1) / (exp_df.sort_read_counts.sum() + 1))/((exp_df.lib_read_counts + 1) / (exp_df.lib_read_counts.sum() + 1)))
exp_df

,aa_substitutions,ddG_fold,ddG_bind,count,n_cells_pelleted,n_plasmids_eluted,Tv,n_amplicons,lib_read_counts,dG_fold,p_fold,dG_bind,Kd_variant,occupancy,p_sort,n_cells_sorted,sort_read_counts,enrichment
0,NaN,0.000000,0.000000,1227,194,0,0,0.0,123,-4.800000,0.999697,-9.000000,2.525085e-07,0.283681,7.806721e-03,7,0,-6.954196
1,Q2A,1.977038,-0.354784,47,72,0,0,0.0,0,-2.822962,0.991550,-9.354784,1.387366e-07,0.418872,4.898593e-02,3,0,0.000000
2,Q2C,1.926444,0.097025,12,0,0,0,0.0,0,-2.873556,0.992236,-8.902975,2.974428e-07,0.251609,3.761467e-03,0,0,0.000000
3,Q2D,2.676374,0.199019,54,0,0,0,0.0,0,-2.123626,0.973002,-8.800981,3.533238e-07,0.220593,1.464829e-03,0,0,0.000000
4,Q2E,1.126964,0.185110,263,0,0,0,0.0,0,-3.673036,0.997975,-8.814890,3.451253e-07,0.224656,1.939488e-03,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
537126,T55Y E56S,2.206998,1.931861,238,0,0,0,0.0,0,-2.593002,0.987592,-7.068139,6.583940e-06,0.014961,4.146893e-17,0,0,0.000000
537127,T55Y E56T,2.441320,1.787153,1216,0,0,0,0.0,0,-2.358680,0.981682,-7.212847,5.157073e-06,0.019022,1.940591e-15,0,0,0.000000
537128,T55Y E56V,2.289906,2.214742,63,0,0,0,0.0,0,-2.510094,0.985755,-6.785258,1.061353e-05,0.009334,8.939990e-21,0,0,0.000000
537129,T55Y E56W,1.240114,2.007181,51,0,0,0,0.0,0,-3.559886,0.997550,-6.992819,7.476554e-06,0.013199,5.761690e-18,0,0,0.000000


In [21]:
import plotly.express as px

fig = px.scatter(exp_df.loc[exp_df.lib_read_counts >= 7], "ddG_bind", "enrichment", color="ddG_fold")
fig.show()

In [22]:
import plotly.express as px

fig = px.scatter(exp_df.loc[exp_df.lib_read_counts >= 10], "p_fold", "occupancy", color="enrichment")
fig.show()

In [96]:
from scipy.optimize import brentq

def find_gate_for_percentile(df, target_fraction, sigma_facs, t_exp):
        """
        Numerically solves for the Binding Threshold that results in 
        sorting exactly `target_fraction` (e.g. 0.01) of the library.
        """
        
        # Pre-calculate log means for speed
        # Signal = P_fold * Occupancy
        # We assume expression gate (t_exp) is fixed/permissive, 
        # and we are tuning the binding gate (t_bind).
        
        # 1. Filter out dead/non-expressing cells first (Optimization)
        # Cells must pass expression gate to even be considered.
        # For simplicity in 'Top 1%', we usually assume we tune the 
        # diagonal or binding axis. Here we tune 't_bind'.
        
        def calculate_fraction_sorted(threshold_candidate):
            # Calculate P_sort for this specific threshold
            # Effective Threshold = max(t_exp, threshold / occupancy)
            
            with np.errstate(divide='ignore', invalid='ignore'):
                req_exp = threshold_candidate / df['occupancy']
                t_eff = np.maximum(t_exp, req_exp)
                
                # Z-score
                mu_log = np.log(df['p_fold'] + 1e-12)
                th_log = np.log(t_eff + 1e-12)
                z = (mu_log - th_log) / sigma_facs
                
                probs = norm.cdf(z)
                probs[np.isinf(t_eff)] = 0.0
                
            # Weighted sum of probabilities / Total Cells
            expected_sorted = np.sum(probs * df['count'])
            total_cells = df['count'].sum()
            return (expected_sorted / total_cells) - target_fraction

        # Use root finding to find the threshold where (Actual - Target) == 0
        # Search range: 0.0001 (very loose) to 2.0 (impossible)
        try:
            optimal_threshold = brentq(calculate_fraction_sorted, 0.00001, 5.0)
        except ValueError:
            print("Warning: Could not converge on exact percentile. Defaulting to 0.5")
            optimal_threshold = 0.5
            
        return optimal_threshold

opt_thresh = find_gate_for_percentile(exp_df, 0.01, params['facs_noise'], params['t_exp'])
opt_thresh

0.38249289307664985